In [2]:
import numpy as np 
import pandas as pd 
df=pd.read_csv("../data/cleaned_telco_churn.csv")
print(df.shape)
print(df.columns)
print(df.dtypes)
print(df.head())

(7032, 21)
Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object
   cus

In [3]:
# Separate features and target

X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn'].map({'No': 0, 'Yes': 1})

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (7032, 19)
y shape: (7032,)

Target distribution:
Churn
0    5163
1    1869
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training set: (5625, 19)
Testing set: (1407, 19)

Training target distribution:
Churn
0    0.734222
1    0.265778
Name: proportion, dtype: float64

Testing target distribution:
Churn
0    0.734186
1    0.265814
Name: proportion, dtype: float64


In [5]:
# Identify numerical and categorical columns

numerical_features = X.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

categorical_features = X.select_dtypes(
    include=['object']
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [6]:
#ColumnTransformer applies StandardScalar for numerical and OneHotEncoder for the categorical at a time
#StandardScalar aligns all the numerical columns on same line cause tenure range is 0-72 
#but totalCharges can range upto 1000 so this makes difficult to train a model so scaling is important 
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

print(preprocessor)

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['SeniorCitizen', 'tenure', 'MonthlyCharges',
                                  'TotalCharges']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['gender', 'Partner', 'Dependents',
                                  'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod'])])


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logistic_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

In [8]:
logistic_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['SeniorCitizen', 'tenure',
                                                   'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['gender', 'Partner',
                                                   'Dependents', 'PhoneService',
                                                   'MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                                                   'TechSupport', 'StreamingTV',
                                                   'StreamingMovies',
                                                   'Contract',
                                                   'PaperlessBilling',
                                                   'PaymentMethod'])])),
                ('model', LogisticRegression(max_iter=1000, random_state=42))])

In [9]:
y_pred = logistic_pipeline.predict(X_test)

y_prob = logistic_pipeline.predict_proba(X_test)[:, 1]

print("Training completed.")
print("Number of predictions:", len(y_pred))

Training completed.
Number of predictions: 1407


In [10]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))
print("ROC-AUC  :", round(roc_auc, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy : 0.8038
Precision: 0.6485
Recall   : 0.5722
F1 Score : 0.608
ROC-AUC  : 0.8359

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1033
           1       0.65      0.57      0.61       374

    accuracy                           0.80      1407
   macro avg       0.75      0.73      0.74      1407
weighted avg       0.80      0.80      0.80      1407



In [11]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# --------------------------------------------------
# 1. Define models
# --------------------------------------------------

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        verbosity=-1,
        random_state=42
    )
}


# --------------------------------------------------
# 2. Train and evaluate every model
# --------------------------------------------------

results = []
trained_models = {}

for name, model in models.items():

    print(f"Training {name}...")

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred_model = pipeline.predict(X_test)
    y_prob_model = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred_model),
        "Precision": precision_score(y_test, y_pred_model),
        "Recall": recall_score(y_test, y_pred_model),
        "F1 Score": f1_score(y_test, y_pred_model),
        "ROC-AUC": roc_auc_score(y_test, y_prob_model)
    })

    trained_models[name] = pipeline


# --------------------------------------------------
# 3. Create comparison table
# --------------------------------------------------

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="ROC-AUC",
    ascending=False
).reset_index(drop=True)

print("\nModel Comparison:")
print(results_df.round(4))

Training Logistic Regression...
Training Decision Tree...
Training Random Forest...
Training XGBoost...
Training LightGBM...

Model Comparison:
                 Model  Accuracy  Precision  Recall  F1 Score  ROC-AUC
0  Logistic Regression    0.8038     0.6485  0.5722    0.6080   0.8359
1              XGBoost    0.7832     0.6075  0.5214    0.5612   0.8301
2        Decision Tree    0.7896     0.6021  0.6150    0.6085   0.8296
3             LightGBM    0.7690     0.5754  0.5000    0.5351   0.8197
4        Random Forest    0.7903     0.6357  0.4947    0.5564   0.8128


c:\Users\sneha\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\sneha\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Create Logistic Regression pipeline
logistic_tuning_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Hyperparameters to test
param_grid = {
    'model__C': [0.01, 0.1, 1, 10, 100]
}

# Grid Search with 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=logistic_tuning_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

# IMPORTANT:
# Only training data is used here
grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation ROC-AUC:")
print(round(grid_search.best_score_, 4))

Best Parameters:
{'model__C': 100}

Best Cross-Validation ROC-AUC:
0.8462


In [13]:
best_logistic_model = grid_search.best_estimator_

In [14]:
y_pred_tuned = best_logistic_model.predict(X_test)

y_prob_tuned = best_logistic_model.predict_proba(X_test)[:, 1]

print("Tuned Logistic Regression")
print("-------------------------")

print("Accuracy :", round(
    accuracy_score(y_test, y_pred_tuned), 4
))

print("Precision:", round(
    precision_score(y_test, y_pred_tuned), 4
))

print("Recall   :", round(
    recall_score(y_test, y_pred_tuned), 4
))

print("F1 Score :", round(
    f1_score(y_test, y_pred_tuned), 4
))

print("ROC-AUC  :", round(
    roc_auc_score(y_test, y_prob_tuned), 4
))

Tuned Logistic Regression
-------------------------
Accuracy : 0.7967
Precision: 0.6317
Recall   : 0.5642
F1 Score : 0.596
ROC-AUC  : 0.8348


In [15]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

# Thresholds we want to test
thresholds = np.arange(0.20, 0.61, 0.05)

threshold_results = []

for threshold in thresholds:

    # Convert probability into prediction
    y_pred_threshold = (
        y_prob >= threshold
    ).astype(int)

    # Calculate metrics
    precision = precision_score(
        y_test,
        y_pred_threshold,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred_threshold,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred_threshold,
        zero_division=0
    )

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })


# Convert results into a DataFrame
threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.round(4).to_string(index=False))

 Threshold  Precision  Recall  F1 Score
      0.20     0.4577  0.8690    0.5996
      0.25     0.4841  0.8155    0.6076
      0.30     0.5108  0.7594    0.6108
      0.35     0.5445  0.7193    0.6198
      0.40     0.5792  0.6845    0.6275
      0.45     0.6046  0.6337    0.6188
      0.50     0.6485  0.5722    0.6080
      0.55     0.6691  0.4866    0.5635
      0.60     0.6792  0.3850    0.4915


In [16]:
# Create prediction dataframe

prediction_df = X_test.copy()

prediction_df["Actual_Churn"] = y_test.values

prediction_df["Churn_Probability"] = y_prob

prediction_df["Predicted_Churn"] = (
    y_prob >= 0.40
).astype(int)

prediction_df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Actual_Churn,Churn_Probability,Predicted_Churn
971,Female,0,Yes,Yes,59,Yes,No,DSL,No,Yes,...,Yes,Yes,Two year,Yes,Credit card (automatic),75.95,4542.35,0,0.017513,0
618,Female,0,No,No,7,Yes,Yes,Fiber optic,No,Yes,...,No,No,Month-to-month,Yes,Bank transfer (automatic),78.55,522.95,0,0.591921,1
4282,Female,0,No,No,54,Yes,No,No,No internet service,No internet service,...,No internet service,No internet service,Two year,No,Mailed check,20.10,1079.45,0,0.004821,0
3715,Female,0,No,No,2,Yes,No,No,No internet service,No internet service,...,No internet service,No internet service,Month-to-month,No,Mailed check,20.65,38.70,1,0.201652,0
4525,Female,0,Yes,No,71,Yes,Yes,Fiber optic,No,Yes,...,Yes,Yes,Two year,Yes,Bank transfer (automatic),105.15,7555.00,0,0.101501,0


In [17]:
def assign_risk(probability):

    if probability < 0.30:
        return "Low"

    elif probability < 0.60:
        return "Medium"

    else:
        return "High"


prediction_df["Risk_Level"] = prediction_df[
    "Churn_Probability"
].apply(assign_risk)

In [21]:
prediction_df["customerID"] = df.loc[
    prediction_df.index, "customerID"
]
prediction_df[
    [
        "customerID",
        "Churn_Probability",
        "Predicted_Churn",
        "Risk_Level"
    ]
].head(10)

,customerID,Churn_Probability,Predicted_Churn,Risk_Level
971,0604-THJFP,0.017513,0,Low
618,4059-IIEBK,0.591921,1,Medium
4282,2228-BZDEE,0.004821,0,Low
3715,2839-RFSQE,0.201652,0,Low
4525,5360-LJCNJ,0.101501,0,Low
445,7752-XUSCI,0.471845,1,Medium
5889,8277-RVRSV,0.026518,0,Low
3381,3530-CRZSB,0.165431,0,Low
1341,2845-HSJCY,0.679600,1,High
5681,0336-KXKFK,0.015525,0,Low


In [25]:
prediction_df["Risk_Level"].value_counts()
#prediction_df["Risk_Level"].value_counts(normalize=True)*100

Risk_Level
Low       851
Medium    344
High      212
Name: count, dtype: int64

In [26]:
prediction_df.groupby(
    ["Risk_Level", "Contract"]
).size().unstack(fill_value=0)

Contract,Month-to-month,One year,Two year
Risk_Level,,,
High,212,0,0
Low,252,272,327
Medium,326,18,0


In [27]:
risk_contract_percentage = pd.crosstab(
    prediction_df["Contract"],
    prediction_df["Risk_Level"],
    normalize="index"
) * 100

risk_contract_percentage.round(2)

Risk_Level,High,Low,Medium
Contract,,,
Month-to-month,26.84,31.90,41.27
One year,0.00,93.79,6.21
Two year,0.00,100.00,0.00


In [28]:
# ============================================================
# POST-ML CUSTOMER RISK ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. BASIC RISK DISTRIBUTION
# ------------------------------------------------------------

print("=" * 60)
print("1. RISK LEVEL DISTRIBUTION")
print("=" * 60)

risk_count = prediction_df["Risk_Level"].value_counts()

risk_percentage = (
    prediction_df["Risk_Level"]
    .value_counts(normalize=True) * 100
)

risk_summary = pd.DataFrame({
    "Customers": risk_count,
    "Percentage": risk_percentage.round(2)
})

print(risk_summary)


# ------------------------------------------------------------
# 2. RISK VS CONTRACT
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("2. RISK LEVEL VS CONTRACT")
print("=" * 60)

risk_contract = pd.crosstab(
    prediction_df["Contract"],
    prediction_df["Risk_Level"]
)

print(risk_contract)

print("\nRisk percentage within each contract:")

risk_contract_pct = pd.crosstab(
    prediction_df["Contract"],
    prediction_df["Risk_Level"],
    normalize="index"
) * 100

print(risk_contract_pct.round(2))


# ------------------------------------------------------------
# 3. RISK VS INTERNET SERVICE
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("3. RISK LEVEL VS INTERNET SERVICE")
print("=" * 60)

risk_internet = pd.crosstab(
    prediction_df["InternetService"],
    prediction_df["Risk_Level"]
)

print(risk_internet)

print("\nRisk percentage within each Internet Service:")

risk_internet_pct = pd.crosstab(
    prediction_df["InternetService"],
    prediction_df["Risk_Level"],
    normalize="index"
) * 100

print(risk_internet_pct.round(2))


# ------------------------------------------------------------
# 4. RISK VS PAYMENT METHOD
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("4. RISK LEVEL VS PAYMENT METHOD")
print("=" * 60)

risk_payment = pd.crosstab(
    prediction_df["PaymentMethod"],
    prediction_df["Risk_Level"]
)

print(risk_payment)

print("\nRisk percentage within each Payment Method:")

risk_payment_pct = pd.crosstab(
    prediction_df["PaymentMethod"],
    prediction_df["Risk_Level"],
    normalize="index"
) * 100

print(risk_payment_pct.round(2))


# ------------------------------------------------------------
# 5. RISK VS TECH SUPPORT
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("5. RISK LEVEL VS TECH SUPPORT")
print("=" * 60)

risk_tech = pd.crosstab(
    prediction_df["TechSupport"],
    prediction_df["Risk_Level"]
)

print(risk_tech)

print("\nRisk percentage within each Tech Support group:")

risk_tech_pct = pd.crosstab(
    prediction_df["TechSupport"],
    prediction_df["Risk_Level"],
    normalize="index"
) * 100

print(risk_tech_pct.round(2))


# ------------------------------------------------------------
# 6. RISK VS ONLINE SECURITY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("6. RISK LEVEL VS ONLINE SECURITY")
print("=" * 60)

risk_security = pd.crosstab(
    prediction_df["OnlineSecurity"],
    prediction_df["Risk_Level"]
)

print(risk_security)

print("\nRisk percentage within each Online Security group:")

risk_security_pct = pd.crosstab(
    prediction_df["OnlineSecurity"],
    prediction_df["Risk_Level"],
    normalize="index"
) * 100

print(risk_security_pct.round(2))


# ------------------------------------------------------------
# 7. RISK VS TENURE
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("7. TENURE BY RISK LEVEL")
print("=" * 60)

tenure_summary = prediction_df.groupby(
    "Risk_Level"
)["tenure"].agg(
    ["count", "mean", "median", "min", "max"]
).round(2)

print(tenure_summary)


# ------------------------------------------------------------
# 8. RISK VS MONTHLY CHARGES
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("8. MONTHLY CHARGES BY RISK LEVEL")
print("=" * 60)

monthly_charges_summary = prediction_df.groupby(
    "Risk_Level"
)["MonthlyCharges"].agg(
    ["count", "mean", "median", "min", "max"]
).round(2)

print(monthly_charges_summary)


# ------------------------------------------------------------
# 9. RISK VS TOTAL CHARGES
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("9. TOTAL CHARGES BY RISK LEVEL")
print("=" * 60)

total_charges_summary = prediction_df.groupby(
    "Risk_Level"
)["TotalCharges"].agg(
    ["count", "mean", "median", "min", "max"]
).round(2)

print(total_charges_summary)


# ------------------------------------------------------------
# 10. HIGH-RISK CUSTOMER COUNT
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("10. HIGH-RISK CUSTOMERS")
print("=" * 60)

high_risk_customers = prediction_df[
    prediction_df["Risk_Level"] == "High"
].copy()

print(
    "Number of High-Risk Customers:",
    len(high_risk_customers)
)


# ------------------------------------------------------------
# 11. TOP HIGH-RISK CUSTOMERS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("11. TOP HIGH-RISK CUSTOMERS")
print("=" * 60)

top_high_risk = high_risk_customers.sort_values(
    "Churn_Probability",
    ascending=False
)

print(
    top_high_risk[
        [
            "customerID",
            "Churn_Probability",
            "Contract",
            "InternetService",
            "PaymentMethod",
            "tenure",
            "MonthlyCharges",
            "TechSupport",
            "OnlineSecurity"
        ]
    ].head(20).to_string(index=False)
)


# ------------------------------------------------------------
# 12. ACTUAL CHURN VS PREDICTED RISK
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("12. ACTUAL CHURN VS RISK LEVEL")
print("=" * 60)

actual_churn_risk = pd.crosstab(
    prediction_df["Risk_Level"],
    prediction_df["Actual_Churn"]
)

print(actual_churn_risk)


print("\nActual churn percentage within each Risk Level:")

actual_churn_pct = pd.crosstab(
    prediction_df["Risk_Level"],
    prediction_df["Actual_Churn"],
    normalize="index"
) * 100

print(actual_churn_pct.round(2))


# ------------------------------------------------------------
# 13. FINAL HIGH-RISK PROFILE
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("13. HIGH-RISK CUSTOMER PROFILE")
print("=" * 60)

high_risk = prediction_df[
    prediction_df["Risk_Level"] == "High"
]

print("\nContract:")
print(high_risk["Contract"].value_counts())

print("\nInternet Service:")
print(high_risk["InternetService"].value_counts())

print("\nPayment Method:")
print(high_risk["PaymentMethod"].value_counts())

print("\nTech Support:")
print(high_risk["TechSupport"].value_counts())

print("\nOnline Security:")
print(high_risk["OnlineSecurity"].value_counts())

print("\nAverage Tenure:")
print(round(high_risk["tenure"].mean(), 2))

print("\nAverage Monthly Charges:")
print(round(high_risk["MonthlyCharges"].mean(), 2))


# ------------------------------------------------------------
# 14. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("14. FINAL ML RISK SUMMARY")
print("=" * 60)

print("Total customers analysed :", len(prediction_df))
print("Low Risk customers       :", (prediction_df["Risk_Level"] == "Low").sum())
print("Medium Risk customers    :", (prediction_df["Risk_Level"] == "Medium").sum())
print("High Risk customers      :", (prediction_df["Risk_Level"] == "High").sum())

print("\nPost-ML analysis completed successfully.")

1. RISK LEVEL DISTRIBUTION
            Customers  Percentage
Risk_Level                       
Low               851       60.48
Medium            344       24.45
High              212       15.07

2. RISK LEVEL VS CONTRACT
Risk_Level      High  Low  Medium
Contract                         
Month-to-month   212  252     326
One year           0  272      18
Two year           0  327       0

Risk percentage within each contract:
Risk_Level       High     Low  Medium
Contract                             
Month-to-month  26.84   31.90   41.27
One year         0.00   93.79    6.21
Two year         0.00  100.00    0.00

3. RISK LEVEL VS INTERNET SERVICE
Risk_Level       High  Low  Medium
InternetService                   
DSL                13  343     126
Fiber optic       199  198     216
No                  0  310       2

Risk percentage within each Internet Service:
Risk_Level        High    Low  Medium
InternetService                      
DSL               2.70  71.16   26.14
Fiber 